# 🗄️ Notebook 02 — Local BLAST+ and Database Management

**What this notebook covers**  
This notebook builds on Notebook 01. There you learned how BLAST works and how the pipeline parses XML results. Here you will learn how to run BLAST *offline*, using a locally installed copy of NCBI BLAST+ and a locally stored database. You will understand why local BLAST is essential for reproducible, regulatory-grade eDNA surveys, and you will see how the pipeline manages databases, caching, and audit trails.

**Prerequisites**  
- You have worked through `01_BLAST_and_Species_Parsing.ipynb`
- Python environment with the project's `requirements.txt` installed
- BLAST+ does **not** need to be installed to run most cells in this notebook — cells that call BLAST will show graceful error messages instead

**You do not need internet access for any cell in this notebook.**

---

## Section 1 — Why local BLAST matters

### The reproducibility problem with web BLAST

When you submit a sequence to NCBI BLAST via the website or the web API, you are searching against the *current* version of the database. The NCBI nucleotide database (`nt`) is updated **daily** — new sequences are added and some entries are revised. This means:

- A sequence you search today might return *Daphnia magna* as the top hit
- The same sequence searched in six months might return *Daphnia pulex* — not because your sequence changed, but because a better reference was added to the database
- If you cannot state exactly which version of the database you searched, your results cannot be independently reproduced

For UK government ecological surveys (Natural England, Environment Agency), **reproducibility is not optional**. A survey report must be re-analysable years later with identical results.

### Local BLAST solves this

| Property | Web BLAST | Local BLAST |
|---|---|---|
| Database version | Changes daily — unknown unless you record it | Fixed — you control the version |
| Speed | 30–120 s per query (API rate limits) | <1 s per query on SSD for small databases |
| Internet required | Yes | No |
| Reproducibility | Requires careful version logging | Guaranteed by locking the database file |
| Cost | Free (public API) | Free (BLAST+ is open source; storage cost only) |
| Rate limits | 3 requests/second (without API key) | None |

### How the pipeline handles this

The pipeline has two BLAST modes, selectable by a flag:
- `--local` → calls `src/local_blast.py` → runs the `blastn` binary on your machine
- (default) → calls `src/blast_query.py` → submits to NCBI via the Entrez API

Both modes produce identical XML output files and use the same XML-based caching, so switching between them does not change any downstream code.

---

## Section 2 — BLAST+ database files

### What the files are

A BLAST+ database is not a single file — it is a set of files that together form an indexed, binary-encoded version of a FASTA sequence file. For a nucleotide database the three core files are:

| File extension | Contents |
|---|---|
| `.nsq` | The **sequence data** — actual nucleotide sequences compressed in a binary format |
| `.nin` | The **index** — maps sequence IDs to their position in the .nsq file (like a book's index) |
| `.nhr` | The **headers** — organism name, accession, and title for each sequence |

For large databases like `nt` (which contains >200 GB of sequence data), BLAST+ splits these into numbered volumes: `nt.00.nsq`, `nt.01.nsq`, etc.

### The .nal alias file

When a database is split across multiple volumes, BLAST+ creates a `.nal` (nucleotide alias) file. This is a plain text file that lists all the volume names. When you pass `-db /data/blast/nt` to blastn, BLAST reads `nt.nal` first, then loads whichever volumes it needs for your query. This means you always refer to the database by a single path, regardless of how many volumes it has.

Example `nt.nal` content:
```
TITLE Nucleotide collection (nt)
DBLIST nt.00 nt.01 nt.02 nt.03
```

### Checking database version with get_db_version()

The function `get_db_version()` in `src/local_blast.py` calls the `blastdbcmd` utility to query a database's metadata — its title and creation date. This is the information the pipeline records in `db_version.json`.

In the cell below, we call it on a fake path to show what happens when no database is installed. This is expected — the graceful fallback is by design.

In [ ]:
import sys
from pathlib import Path

# Add project root to path so we can import from src/
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.local_blast import get_db_version

# Try to get version info for a fake database path
# This will return a graceful fallback string — not an exception
fake_db_path = "/data/blast/nt"      # this path does not exist on this machine
version_info = get_db_version(fake_db_path)

print(f"Database path   : {fake_db_path}")
print(f"Version returned: {version_info}")
print()
print("This output is expected — 'blastdbcmd' is not installed, so the function")
print("returns a descriptive fallback string instead of raising an exception.")
print("When a real BLAST+ database is installed, this would return something like:")
print("  'Database: Nucleotide collection (nt) | Date: Feb 14, 2025 | 75,259,152 sequences'")

---

## Section 3 — Checking whether BLAST+ is installed

Before running local BLAST, the pipeline checks whether the `blastn` binary is on your system's PATH. It uses Python's `shutil.which()` function — the same mechanism your terminal uses when you type a command.

The cell below does this check interactively.

In [ ]:
import shutil  # standard library — no install needed

# shutil.which() searches PATH for the given binary, exactly like the shell does
# Returns the full path if found, None if not found
blastn_path = shutil.which("blastn")

if blastn_path:
    print(f"blastn found at: {blastn_path}")
    print("You are ready to run local BLAST+.")
else:
    print("blastn is not installed (not found on PATH).")
    print()
    print("To install BLAST+:")
    print()
    print("  Ubuntu / Debian (including GitHub Codespaces):")
    print("    sudo apt-get install -y ncbi-blast+")
    print()
    print("  macOS (Homebrew):")
    print("    brew install blast")
    print()
    print("  Windows:")
    print("    Download the installer from:")
    print("    https://www.ncbi.nlm.nih.gov/books/NBK153387/")
    print()
    print("  GitHub Codespaces (devcontainer):")
    print("    BLAST+ is pre-installed if you open the project in the devcontainer.")
    print("    Press Ctrl+Shift+P and select 'Dev Containers: Reopen in Container'.")

---

## Section 4 — The caching system

### How caching works

Running BLAST on a database of millions of sequences takes time — 10 seconds to several minutes per query, depending on the database size and your hardware. If the pipeline crashes after 50 of 100 queries, you would not want to re-run all 100 from scratch.

The pipeline uses **file-based caching**: each query writes its result to a file named `{query_id}.xml` in the results directory. When the pipeline starts, it checks whether a non-empty XML file already exists for each query. If it does, the query is skipped. This means:

- You can interrupt and resume a run without losing work
- You can add new sequences to a FASTA file without re-running existing ones
- Running the pipeline twice on the same input is safe and fast

The caching check in `src/local_blast.py` is a single `if` statement:
```python
if xml_path.exists() and xml_path.stat().st_size > 0:
    log.info(f"{query_id} — cached XML found, skipping")
    return query_id
```

### Demonstrating the caching contract

The cell below creates a temporary directory, writes a fake XML file into it, and shows that `parse_all_results` finds it — just as it would find a cached real BLAST result.

In [ ]:
import tempfile
import textwrap
from pathlib import Path

from src.parser import parse_all_results

# Minimal valid BLAST XML — the exact structure the parser expects
CACHED_XML = textwrap.dedent("""\
<?xml version="1.0"?>
<BlastOutput>
  <BlastOutput_program>blastn</BlastOutput_program>
  <BlastOutput_version>BLASTN 2.14.0+</BlastOutput_version>
  <BlastOutput_reference>Reference</BlastOutput_reference>
  <BlastOutput_db>/data/blast/nt</BlastOutput_db>
  <BlastOutput_query-ID>Query_1</BlastOutput_query-ID>
  <BlastOutput_query-def>CACHED_QUERY 16S_rRNA_cached_result</BlastOutput_query-def>
  <BlastOutput_query-len>410</BlastOutput_query-len>
  <BlastOutput_param>
    <Parameters>
      <Parameters_matrix></Parameters_matrix>
      <Parameters_expect>10</Parameters_expect>
      <Parameters_sc-match>1</Parameters_sc-match>
      <Parameters_sc-mismatch>-3</Parameters_sc-mismatch>
      <Parameters_gap-open>5</Parameters_gap-open>
      <Parameters_gap-extend>2</Parameters_gap-extend>
      <Parameters_filter>L;m;</Parameters_filter>
    </Parameters>
  </BlastOutput_param>
  <BlastOutput_iterations>
    <Iteration>
      <Iteration_iter-num>1</Iteration_iter-num>
      <Iteration_query-ID>Query_1</Iteration_query-ID>
      <Iteration_query-def>CACHED_QUERY 16S_rRNA_cached_result</Iteration_query-def>
      <Iteration_query-len>410</Iteration_query-len>
      <Iteration_hits>
        <Hit>
          <Hit_num>1</Hit_num>
          <Hit_id>gi|1000|gb|AB001339.1|</Hit_id>
          <Hit_def>Microcystis aeruginosa strain NIES-843 16S ribosomal RNA gene</Hit_def>
          <Hit_accession>AB001339</Hit_accession>
          <Hit_len>418</Hit_len>
          <Hit_hsps>
            <Hsp>
              <Hsp_num>1</Hsp_num>
              <Hsp_bit-score>800</Hsp_bit-score>
              <Hsp_score>800</Hsp_score>
              <Hsp_evalue>1e-195</Hsp_evalue>
              <Hsp_query-from>1</Hsp_query-from>
              <Hsp_query-to>410</Hsp_query-to>
              <Hsp_hit-from>1</Hsp_hit-from>
              <Hsp_hit-to>406</Hsp_hit-to>
              <Hsp_query-frame>1</Hsp_query-frame>
              <Hsp_hit-frame>1</Hsp_hit-frame>
              <Hsp_identity>403</Hsp_identity>
              <Hsp_positive>403</Hsp_positive>
              <Hsp_gaps>0</Hsp_gaps>
              <Hsp_align-len>406</Hsp_align-len>
              <Hsp_qseq>ACGT</Hsp_qseq>
              <Hsp_hseq>ACGT</Hsp_hseq>
              <Hsp_midline>||||</Hsp_midline>
            </Hsp>
          </Hit_hsps>
        </Hit>
      </Iteration_hits>
      <Iteration_stat>
        <Statistics>
          <Statistics_db-num>1000</Statistics_db-num>
          <Statistics_db-len>1000000</Statistics_db-len>
          <Statistics_hsp-len>0</Statistics_hsp-len>
          <Statistics_eff-space>0</Statistics_eff-space>
          <Statistics_kappa>0.041</Statistics_kappa>
          <Statistics_lambda>0.267</Statistics_lambda>
          <Statistics_entropy>0.14</Statistics_entropy>
        </Statistics>
      </Iteration_stat>
    </Iteration>
  </BlastOutput_iterations>
</BlastOutput>
""")

with tempfile.TemporaryDirectory() as tmpdir:
    xml_dir = Path(tmpdir) / "xml"
    xml_dir.mkdir()

    # Write the fake XML as if it were a cached BLAST result
    cached_file = xml_dir / "CACHED_QUERY.xml"
    cached_file.write_text(CACHED_XML, encoding="utf-8")

    print(f"Cached XML file written: {cached_file.name}")
    print(f"File size: {cached_file.stat().st_size} bytes")
    print()

    # The caching check: does a non-empty XML already exist?
    is_cached = cached_file.exists() and cached_file.stat().st_size > 0
    print(f"Would pipeline skip this query? {is_cached}  ← True means: cache hit, skip BLAST")
    print()

    # parse_all_results reads it exactly as it would a real BLAST XML
    hits = parse_all_results(xml_dir, max_hits=5, evalue_threshold=1e-10)
    print(f"Hits parsed from cached file: {len(hits)}")
    for h in hits:
        print(f"  query={h.query_id}  species={h.species}  identity={h.identity_pct}%")

print()
print("The caching contract: parse_all_results reads whatever .xml files are present.")
print("It does not care whether they were written by local BLAST, web BLAST, or by hand.")

---

## Section 5 — Database selection guide

Which database you search against is just as important as which BLAST program you run. For UK freshwater eDNA, the choice depends on the gene you amplified.

| Database | What it contains | Best for | Size | Notes |
|---|---|---|---|---|
| **nt** (NCBI nucleotide) | All public nucleotide sequences — everything | Universal fallback; great crested newt; fish | >200 GB | Most complete; slowest; updated daily |
| **16S\_ribosomal\_RNA** (NCBI) | All 16S rRNA sequences | Cyanobacteria (e.g. *Microcystis*, *Anabaena*); bloom monitoring | ~4 GB | Much faster than nt; pre-formatted by NCBI |
| **ITS\_RefSeq\_Fungi** | Fungal ITS1/ITS2 from RefSeq | Aquatic fungi; chytrid fungi (*Batrachochytrium dendrobatidis* in amphibian surveys) | ~1 GB | Not relevant for standard macrophyte/invertebrate surveys |
| **SILVA 138 SSU** | Curated 16S/18S rRNA sequences | Algae (18S); protists; eukaryotic phytoplankton | ~8 GB | Better curated than nt for 18S; needs manual formatting with `makeblastdb` |

### Recommendations for UK pond surveys

| Survey target | Gene | Database |
|---|---|---|
| Great crested newt | 12S rRNA | nt (the 12S target requires the full database) |
| Cyanobacteria bloom | 16S rRNA | 16S\_ribosomal\_RNA (fast, curated) |
| Green algae / diatoms | 18S rRNA | SILVA 138 SSU (better 18S coverage than nt) |
| Macroinvertebrates | COI | nt or BOLD (Barcode of Life Database, requires separate tool) |
| Aquatic macrophytes | trnL intron | nt (trnL coverage is sparse in specialised databases) |

---

## Section 6 — Running with --local (full command walkthrough)

The pipeline is run from the command line. The full command for a local BLAST run looks like this:

```bash
python pipeline.py \
  --fasta data/sample_sequences.fasta \
  --local \
  --db-path /data/blast/nt \
  --evalue 1e-10 \
  --max-hits 20 \
  --threads 4 \
  --output data/results/survey_run_01
```

Here is what each flag does:

| Flag | Value | What it does |
|---|---|---|
| `--fasta` | `data/sample_sequences.fasta` | Path to your eDNA sequences in FASTA format |
| `--local` | (no value — boolean flag) | Switch to local BLAST+ mode; without this flag the pipeline uses the NCBI web API |
| `--db-path` | `/data/blast/nt` | Path to the local BLAST database, without file extension |
| `--evalue` | `1e-10` | Discard any alignment with E-value worse than 1×10⁻¹⁰ |
| `--max-hits` | `20` | Return at most 20 alignments per query — the parser then applies the per-query hit limit |
| `--threads` | `4` | Run 4 BLAST processes in parallel; use 1 for spinning hard drives |
| `--output` | `data/results/survey_run_01` | Directory where results (XML files, CSVs, PDF report) are written |

### Output files produced

After the pipeline runs, the output directory contains:

```
survey_run_01/
├── xml/
│   ├── SEQ001.xml          ← one per query — cached on re-runs
│   ├── SEQ002.xml
│   └── ...
├── blast_hit_table.csv     ← all hits with identity, evalue, resolved species
├── biodiversity_summary.csv
├── db_version.json         ← database audit trail (see Section 7)
└── eDNA_Report.pdf
```

---

## Section 7 — db_version.json — your audit trail

### Why it matters

Whenever the pipeline runs in local mode, it writes a `db_version.json` file to the output directory. This file records:
- The exact path to the database that was searched
- The database title and creation date (as reported by `blastdbcmd`)
- The BLAST program used
- The timestamp of the run

This is your **taxonomic audit trail**. For a Natural England licensing application, or an Environment Agency habitat survey, you must be able to state precisely:

> *"Sequences were searched against the NCBI nucleotide database version dated 14 February 2025, using BLASTN 2.14.0+, with an e-value threshold of 1×10⁻¹⁰."*

Without `db_version.json`, you cannot make that statement. Future re-analysis might use a different database version and produce different species identifications, undermining the survey's evidential value.

If `db_version.json` is absent from a submission, a regulatory reviewer has no way to verify which database was used. This would typically require the survey to be repeated — an expensive outcome.

The cell below creates a mock version dict and shows what the file looks like.

In [ ]:
import json
from datetime import datetime, timezone

# This is what stamp_db_version() in src/local_blast.py writes
# when a real BLAST+ database is present
mock_version = {
    "db_path": "/data/blast/nt",
    "db_version": (
        "Database: Nucleotide collection (nt) | "
        "Date: Feb 14, 2025  1:04 PM | "
        "75,259,152 sequences | "
        "total residues: 282,812,267,832"
    ),
    "program": "blastn",
    "stamped_at": "2026-05-23T08:00:00Z",   # ISO 8601 UTC timestamp
}

# json.dumps with indent=2 produces the same pretty-printed format
# as the real stamp_db_version() function
print("Contents of db_version.json:")
print()
print(json.dumps(mock_version, indent=2))

In [ ]:
# Show how to read db_version.json back and check it programmatically
# This is useful for a post-processing quality check

import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmpdir:
    version_file = Path(tmpdir) / "db_version.json"

    # Write the mock version file (mimicking what the pipeline does)
    version_file.write_text(json.dumps(mock_version, indent=2), encoding="utf-8")

    # Read it back
    loaded = json.loads(version_file.read_text(encoding="utf-8"))

# Run a simple audit check
print("Audit check on db_version.json:")
print()

# Check: was the database nt? (not a specialist subset database)
db_is_nt = "/nt" in loaded["db_path"] or loaded["db_path"].endswith("nt")
print(f"  Database path      : {loaded['db_path']}")
print(f"  Is nt database     : {db_is_nt}")

# Check: program is blastn (not tblastx or protein blast)
print(f"  Program            : {loaded['program']}")
print(f"  Correct program    : {loaded['program'] == 'blastn'}")

# Check: stamped at (run date)
print(f"  Run timestamp      : {loaded['stamped_at']}")
print()
print("Version string (for inclusion in regulatory report):")
print(f"  {loaded['db_version']}")

---

## Section 8 — Exercises

Two exercises to consolidate your understanding of local BLAST and database management.

### Exercise 1 — Choose the right database

For each of the following survey scenarios, identify the most appropriate BLAST database from the options in Section 5. Give a one-sentence justification for each choice.

| Survey | Target organism | Gene amplified | Your database choice | Justification |
|---|---|---|---|---|
| A | Great crested newt in a Cheshire pond | 12S rRNA | ? | ? |
| B | Cyanobacteria bloom in a Norfolk broad | 16S rRNA | ? | ? |
| C | Green algae and diatoms in a Welsh upland lake | 18S rRNA | ? | ? |
| D | Aquatic invertebrate survey (mayflies, stoneflies) | COI | ? | ? |

**Bonus:** For survey B, what specific cyanobacterium would you hope *not* to detect in a drinking water reservoir, and why?

### Exercise 2 — Missing db_version.json

You receive a results folder from a colleague. It contains `blast_hit_table.csv` and `biodiversity_summary.csv`, but `db_version.json` is missing.

Answer the following:

1. **What information is missing?** List the four fields that `db_version.json` normally records.

2. **What is the practical consequence?** The results show *Triturus cristatus* detected at a survey site. You want to include this in a Natural England licence application. Why does the missing file create a problem for the submission?

3. **Can it be recovered?** Your colleague ran the analysis three months ago on a machine that is now decommissioned. The database was `nt` from NCBI but the exact date is unknown. Suggest two steps you would take to document this as thoroughly as possible without being able to recover the original file.

4. **How to prevent it next time?** Look at the `run_local_blast()` function signature in `src/local_blast.py`. Which parameter, if provided, causes `db_version.json` to be written automatically? Write the one-line code change you would add to `pipeline.py` to ensure the version file is always written.